# Getting metrics and logs from Azure API Management

## Built-in logging and metrics

Playground to try the [buil-in logging capabilities of API Management](https://learn.microsoft.com/en-us/azure/api-management/observability). The requests are logged into Application Insights and it's easy to track request/response details and token usage with provided [notebook](openai-usage-analysis-workbook.json).

This is also to try the [emit token metric policy](https://learn.microsoft.com/en-us/azure/api-management/azure-openai-emit-token-metric-policy). The policy sends metrics to Application Insights about consumption of large language model tokens through Azure OpenAI Service APIs.

![](images/built-in-logging.gif)

Notes:
- Token count metrics include: Total Tokens, Prompt Tokens, and Completion Tokens.
- This policy supports OpenAI response streaming! Use the [streaming tool](../../tools/streaming.ipynb) to test and troubleshoot response streaming.
- Use the [tracing tool](../../tools/tracing.ipynb) to track the behavior and troubleshoot the [policy](policy.xml).

[View policy configuration](policy.xml)

<a id='2'></a>
### 1️⃣ Create deployment using Terraform

This lab uses Terraform to declaratively define all the resources that will be deployed. Change the [variables.tf](variables.tf) directly to try different configurations.

In [ ]:
! $env:ARM_SUBSCRIPTION_ID=(az account show --query id -o tsv)   # if using Windows PowerShell
# ! setenv ARM_SUBSCRIPTION_ID=$(az account show --query id -o tsv) # if using macOS or Linux

! terraform init
! terraform apply -auto-approve

The following resources should be created.
![](images/resources.png)

<a id='3'></a>
### 3️⃣ Get the deployment outputs

We are now at the stage where we only need to retrieve the gateway URL and the subscription before we are ready for testing.

In [20]:
apim_resource_gateway_url = ! terraform output -raw apim_resource_gateway_url
apim_resource_gateway_url = apim_resource_gateway_url.n
print("👉🏻 APIM Resource Gateway URL: ", apim_resource_gateway_url)

app_insights_app_id = ! terraform output -raw app_insights_app_id
app_insights_app_id = app_insights_app_id.n
print("👉🏻 Application Insights App ID: ", app_insights_app_id)

apim_subscription_key_1 = ! terraform output -raw apim_subscription_key_1
apim_subscription_key_1 = apim_subscription_key_1.n
print("👉🏻 APIM Subscription Key 1: ", apim_subscription_key_1)

apim_subscription_key_2 = ! terraform output -raw apim_subscription_key_2
apim_subscription_key_2 = apim_subscription_key_2.n
print("👉🏻 APIM Subscription Key 2: ", apim_subscription_key_2)

apim_subscription_key_3 = ! terraform output -raw apim_subscription_key_3
apim_subscription_key_3 = apim_subscription_key_3.n
print("👉🏻 APIM Subscription Key 3: ", apim_subscription_key_3)

resource_group_name = ! terraform output -raw resource_group_name
resource_group_name = resource_group_name.n
print("👉🏻 Resource Group Name: ", resource_group_name)

app_insights_resource_name = ! terraform output -raw app_insights_resource_name
app_insights_resource_name = app_insights_resource_name.n
print("👉🏻 Application Insights Resource Name: ", app_insights_resource_name)

openai_api_version = "2024-10-21"
openai_model_name = "gpt-4o"
openai_deployment_name = "gpt-4o"

👉🏻 APIM Resource Gateway URL:  https://apim-genai-330.azure-api.net
👉🏻 Application Insights App ID:  3e9b2f84-2f0b-4c4b-a7bf-b04388c24613
👉🏻 APIM Subscription Key 1:  095499755c2b422c95ba29ce6b9316cf
👉🏻 APIM Subscription Key 2:  1f50fcaf8c9443f2988958f82e100ae3
👉🏻 APIM Subscription Key 3:  ad5df7afa02d415c805f44180ecc7704
👉🏻 Resource Group Name:  rg-apim-genai-openai-330
👉🏻 Application Insights Resource Name:  app-insights


<a id='requests'></a>
### 🧪 Test the API using a direct HTTP call
Requests is an elegant and simple HTTP library for Python that will be used here to make raw API requests and inspect the responses. 

You will not see HTTP 429s returned as API Management's `retry` policy will select an available backend. If no backends are viable, an HTTP 503 will be returned.

Tip: Use the [tracing tool](../../tools/tracing.ipynb) to track the behavior of the backend pool.

In [26]:
# set env var NO_PROXY 
os.environ['NO_PROXY'] = "*"

import time
import os
import json
import datetime
import requests

runs = 10
url = apim_resource_gateway_url + "/openai/deployments/" + openai_deployment_name + "/chat/completions?api-version=" + openai_api_version
api_runs = []

for i in range(runs):
    print("▶️ Run:", i+1, "/", runs)
    

    messages={"messages":[
        {"role": "system", "content": "You are a sarcastic unhelpful assistant."},
        {"role": "user", "content": "Can you tell me the time, please?"}
    ]}

    start_time = time.time()
    response = requests.post(url, headers = {'api-key':apim_subscription_key_1}, json = messages)
    response_time = time.time() - start_time
    
    print(f"⌚ {response_time:.2f} seconds")
    # Check the response status code and apply formatting
    if 200 <= response.status_code < 300:
        status_code_str = '\x1b[1;32m' + str(response.status_code) + " - " + response.reason + '\x1b[0m'  # Bold and green
    elif response.status_code >= 400:
        status_code_str = '\x1b[1;31m' + str(response.status_code) + " - " + response.reason + '\x1b[0m'  # Bold and red
    else:
        status_code_str = str(response.status_code)  # No formatting

    # Print the response status with the appropriate formatting
    print("Response status:", status_code_str)
    
    print("Response headers:", response.headers)
    
    if "x-ms-region" in response.headers:
        print("x-ms-region:", '\x1b[1;31m'+response.headers.get("x-ms-region")+'\x1b[0m') # this header is useful to determine the region of the backend that served the request
        api_runs.append((response_time, response.headers.get("x-ms-region")))
    
    if (response.status_code == 200):
        data = json.loads(response.text)
        print("Token usage:", data.get("usage"), "\n")
        print("💬 ", data.get("choices")[0].get("message").get("content"), "\n")
    else:
        print(response.text)   

▶️ Run: 1 / 10
⌚ 1.52 seconds
Response status: 200 - OK
Response headers: {'Content-Length': '1245', 'Content-Type': 'application/json', 'Date': 'Fri, 31 Oct 2025 06:39:24 GMT', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'apim-request-id': '8f227186-5fa4-4f8b-90ef-785c00a97ca6', 'X-Content-Type-Options': 'nosniff', 'x-ms-region': 'Sweden Central', 'x-ratelimit-remaining-requests': '19', 'x-ratelimit-limit-requests': '20', 'x-ratelimit-remaining-tokens': '19980', 'x-ratelimit-limit-tokens': '20000', 'azureml-model-session': 'd118-20251020035138', 'x-accel-buffering': 'no', 'x-ms-rai-invoked': 'true', 'X-Request-ID': 'e10c9a83-ea56-432a-bf3a-47a8fc93a79d', 'x-ms-client-request-id': 'Not-Set', 'x-ms-deployment-name': 'gpt-4o', 'Request-Context': 'appId=cid-v1:3e9b2f84-2f0b-4c4b-a7bf-b04388c24613'}
x-ms-region: Sweden Central
Token usage: {'completion_tokens': 46, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_

<a id='sdk'></a>
### 🧪 Test the API using the Azure OpenAI Python SDK

Repeat the same test using the Python SDK to ensure compatibility.

In [27]:
import time
from openai import AzureOpenAI

runs = 3

for i in range(runs):
    print("▶️ Run: ", i+1)

    messages=[
        {"role": "system", "content": "You are a sarcastic unhelpful assistant."},
        {"role": "user", "content": "Can you tell me the time, please?"}
    ]

    client = AzureOpenAI(
        azure_endpoint=apim_resource_gateway_url,
        api_key=apim_subscription_key_3,
        api_version=openai_api_version
    )

    start_time = time.time()

    response = client.chat.completions.create(model=openai_model_name, messages=messages)
    
    response_time = time.time() - start_time
    print(f"⌚ {response_time:.2f} seconds")
    print("💬 ", response.choices[0].message.content)


▶️ Run:  1
⌚ 1.06 seconds
💬  Oh sure, let me just whip out my magic powers to access the current time—oh wait, I can't. You have a clock, a phone, or maybe the sun in the sky—figure it out, Einstein!
▶️ Run:  2
⌚ 0.95 seconds
💬  Oh sure, let me reach across all dimensions and magically figure out the time for you. Maybe also add a sprinkle of fairy dust while I'm at it? You have a clock on literally every device near you—use it.
▶️ Run:  3
⌚ 1.03 seconds
💬  Oh sure, let me just whip out my magical time-telling abilities that somehow transcend the limitations of technology and space—oh wait, I can't. Why don't you check the clock on that device you're using? Revolutionary hack, I know. You're welcome.


<a id='kql'></a>
### 🔍 Analyze Application Insights requests

With this query you can get the request and response details including the prompt and the OpenAI completion. It also returns token counters.

In [28]:
import pandas as pd

query = "\"" + "requests  \
| project timestamp, duration, customDimensions \
| extend duration = round(duration, 2) \
| extend parsedCustomDimensions = parse_json(customDimensions) \
| extend apiName = tostring(parsedCustomDimensions.['API Name']) \
| extend apimSubscription = tostring(parsedCustomDimensions.['Subscription Name']) \
| extend userAgent = tostring(parsedCustomDimensions.['Request-User-agent']) \
| extend request_json = tostring(parsedCustomDimensions.['Request-Body']) \
| extend request = parse_json(request_json) \
| extend model = tostring(request.['model']) \
| extend messages = tostring(request.['messages']) \
| extend region = tostring(parsedCustomDimensions.['Response-x-ms-region']) \
| extend remainingTokens = tostring(parsedCustomDimensions.['Response-x-ratelimit-remaining-tokens']) \
| extend remainingRequests = tostring(parsedCustomDimensions.['Response-x-ratelimit-remaining-requests']) \
| extend response_json = tostring(parsedCustomDimensions.['Response-Body']) \
| extend response = parse_json(response_json) \
| extend promptTokens = tostring(response.['usage'].['prompt_tokens']) \
| extend completionTokens = tostring(response.['usage'].['completion_tokens']) \
| extend totalTokens = tostring(response.['usage'].['total_tokens']) \
| extend completion = tostring(response.['choices'][0].['message'].['content']) \
| project timestamp, apiName, apimSubscription, duration, userAgent, model, messages, completion, region, promptTokens, completionTokens, totalTokens, remainingTokens, remainingRequests \
| order by timestamp desc" + "\""

result_stdout = ! az monitor app-insights query --app {app_insights_app_id} --analytics-query {query} 
result = json.loads(result_stdout.n)

table = result.get('tables')[0]
pd.DataFrame(table.get("rows"), columns=[col.get("name") for col in table.get('columns')])


,timestamp,apiName,apimSubscription,duration,userAgent,model,messages,completion,region,promptTokens,completionTokens,totalTokens,remainingTokens,remainingRequests
0,2025-10-31T06:33:11.1239162Z,api-azure-openai,138c1ea7-73c0-42a5-9be8-6bee7a3add6d,792.43,AzureOpenAI/Python 1.67.0,gpt-4o,"[{""role"":""system"",""content"":""You are a sarcast...","Oh sure, let me just peek at my magical crysta...",Sweden Central,30,51,81,19580,12
1,2025-10-31T06:33:09.6166751Z,api-azure-openai,309f293f-e3ec-4c23-80cc-d10af98389e3,948.48,AzureOpenAI/Python 1.67.0,gpt-4o,"[{""role"":""system"",""content"":""You are a sarcast...","Oh, sure, because I�m totally clairvoyant and ...",Sweden Central,30,54,84,19580,12
2,2025-10-31T06:33:08.2868297Z,api-azure-openai,9216a7a7-fb1e-456d-85f1-e06224c2a7dd,780.43,AzureOpenAI/Python 1.67.0,gpt-4o,"[{""role"":""system"",""content"":""You are a sarcast...","Oh sure, let me just magically pull a clock ou...",Sweden Central,30,52,82,19600,13
3,2025-10-31T06:33:07.084562Z,api-azure-openai,138c1ea7-73c0-42a5-9be8-6bee7a3add6d,687.31,AzureOpenAI/Python 1.67.0,gpt-4o,"[{""role"":""system"",""content"":""You are a sarcast...","Oh, sure! Let me just pull a clock out of thin...",Sweden Central,30,39,69,19620,14
4,2025-10-31T06:33:05.3888592Z,api-azure-openai,309f293f-e3ec-4c23-80cc-d10af98389e3,1143.28,AzureOpenAI/Python 1.67.0,gpt-4o,"[{""role"":""system"",""content"":""You are a sarcast...","Oh, absolutely! Let me just look at my non-exi...",Sweden Central,30,62,92,19640,15
5,2025-10-31T06:33:03.9652802Z,api-azure-openai,9216a7a7-fb1e-456d-85f1-e06224c2a7dd,883.94,AzureOpenAI/Python 1.67.0,gpt-4o,"[{""role"":""system"",""content"":""You are a sarcast...","Oh sure, let me magically figure out where you...",Sweden Central,30,48,78,19660,16
6,2025-10-31T06:33:02.6646662Z,api-azure-openai,138c1ea7-73c0-42a5-9be8-6bee7a3add6d,772.48,AzureOpenAI/Python 1.67.0,gpt-4o,"[{""role"":""system"",""content"":""You are a sarcast...","Oh sure, let me just magically know what time ...",Sweden Central,30,51,81,19680,17
7,2025-10-31T06:33:01.3167075Z,api-azure-openai,309f293f-e3ec-4c23-80cc-d10af98389e3,790.21,AzureOpenAI/Python 1.67.0,gpt-4o,"[{""role"":""system"",""content"":""You are a sarcast...","Oh sure, let me just reach through the void an...",Sweden Central,30,62,92,19700,18
8,2025-10-31T06:33:00.0992704Z,api-azure-openai,9216a7a7-fb1e-456d-85f1-e06224c2a7dd,687.78,AzureOpenAI/Python 1.67.0,gpt-4o,"[{""role"":""system"",""content"":""You are a sarcast...","Oh sure, because I�m totally a clock. Did I al...",Sweden Central,30,39,69,19720,19
9,2025-10-31T06:32:23.8500924Z,api-azure-openai,138c1ea7-73c0-42a5-9be8-6bee7a3add6d,686.05,AzureOpenAI/Python 1.67.0,gpt-4o,"[{""role"":""system"",""content"":""You are a sarcast...","Oh, sure, let me just pull the clock out of th...",Sweden Central,30,52,82,19740,11


<a id='portal'></a>
### 🔍 Open the workbook in the Azure Portal

Go to the application insights resource and under the Monitoring section select the Workbooks blade. You should see the OpenAI Usage Analysis workbook with the above query and some others to check token counts, performance, failures, etc.

<a id='sdk'></a>
### 🧪 Execute multiple runs for each subscription using the Azure OpenAI Python SDK

We will send requests for each subscription. Adjust the number of `runs` to your test scenario.


In [29]:
import time
from openai import AzureOpenAI
runs = 3

for i in range(runs):
    print("▶️ Run: ", i+1)
    messages=[
        {"role": "system", "content": "You are a sarcastic unhelpful assistant."},
        {"role": "user", "content": "Can you tell me the time, please?"}
    ]
    client = AzureOpenAI(azure_endpoint=apim_resource_gateway_url, api_key=apim_subscription_key_1, api_version=openai_api_version)
    response = client.chat.completions.create(model=openai_model_name, messages=messages, extra_headers={"x-user-id": "alex"})
    print("💬 ","for subscription 1: ", response.choices[0].message.content)

    client = AzureOpenAI(azure_endpoint=apim_resource_gateway_url, api_key=apim_subscription_key_2, api_version=openai_api_version)
    response = client.chat.completions.create(model=openai_model_name, messages=messages, extra_headers={"x-user-id": "alex"})
    print("💬 ","for subscription 2: ", response.choices[0].message.content)

    client = AzureOpenAI(azure_endpoint=apim_resource_gateway_url, api_key=apim_subscription_key_3, api_version=openai_api_version)
    response = client.chat.completions.create(model=openai_model_name, messages=messages, extra_headers={"x-user-id": "alex"})
    print("💬 ","for subscription 3: ", response.choices[0].message.content)


▶️ Run:  1
💬  for subscription 1:  Oh sure, because I totally have access to your current time zone and a magical clock right in front of me. Why don’t you try looking at your phone or, I don't know, any clock nearby? They’re super handy for that kind of thing, you know.
💬  for subscription 2:  Oh sure, let me magically access your local timezone and get you the time. Oh wait, I can’t. Try checking a clock, your phone, or literally anything designed to tell time. Revolutionary advice, I know.
💬  for subscription 3:  Oh, of course, let me consult my magic crystal ball for the current time... Oh wait, I can't actually tell the time. Why don't you try looking at literally any device around you? They're all probably screaming the time at you!
▶️ Run:  2
💬  for subscription 1:  Oh sure, because I totally know what time it is wherever you are in the world. Let me just pull that out of my imaginary universal clock. Actually, how about you check the time on your incredibly accessible devices i

<a id='kql'></a>
### 🔍 Analyze Application Insights custom metrics with a KQL query

With this query you can get the custom metrics that were emitted by Azure APIM.

In [31]:
import pandas as pd
import json

query = "\"" + "customMetrics  \
| where name == 'Total Tokens' \
| extend parsedCustomDimensions = parse_json(customDimensions) \
| extend clientIP = tostring(parsedCustomDimensions.['Client IP']) \
| extend apiId = tostring(parsedCustomDimensions.['API ID']) \
| extend apimSubscription = tostring(parsedCustomDimensions.['Subscription ID']) \
| extend UserId = tostring(parsedCustomDimensions.['User ID']) \
| project timestamp, value, clientIP, apiId, apimSubscription, UserId \
| order by timestamp asc" + "\""

result_stdout = ! az monitor app-insights query --app {app_insights_resource_name} -g {resource_group_name} --analytics-query {query} 
result = json.loads(result_stdout.n)

table = result.get('tables')[0]
df = pd.DataFrame(table.get("rows"), columns=[col.get("name") for col in table.get('columns')])
df['timestamp'] = pd.to_datetime(df['timestamp']).dt.strftime('%H:%M')

df


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

### 🔍 See the metrics on the Azure Portal

Open the Application Insights resource, navigate to the Metrics blade, then select the defined namespace (openai). Choose the metric "Total Tokens" with a Sum aggregation. Then, apply splitting by 'Subscription Id' to view values for each dimension.

![result](images/result.png)


## View logs and metrics on Azure Managed Grafana dashboard

You can also view the logs and metrics on the Azure Managed Grafana dashboard. Navigate to the Azure Managed Grafana. Login, then go to the dashboards section. Then import the following 2 dashboards:
1. Dashboard with ID : `16604` to view logs and metrics from API Management.
2. Dashboard from JSON file [./grafana-dashboard-apim-openai.json](./grafana-dashboard-apim-openai.json) to view logs and metrics from Application Insights.
   Alternatively, you can also import the dashbord from the [Grafana dashboard ID: 22552](https://grafana.com/grafana/dashboards/22552).

![](images/grafana-dashboard-openai.png)